In [0]:
%sql
create schema synth_prac.bronze;
create schema synth_prac.silver;
create schema synth_prac.gold;

In [0]:
from pyspark.sql import functions as F

orders_df = spark.read.parquet("/Volumes/synth_prac/data/brz_data/orders_04-21.parquet").withColumn("ingested_time", F.current_timestamp())\
    .write.format("delta").partitionBy("customer_id").mode("append").saveAsTable("synth_prac.bronze.bronze_orders")

In [0]:
%sql
select * from synth_prac.bronze.bronze_orders;

In [0]:
from pyspark.sql import functions as F

last_processed = spark.sql("""
                           select * from synth_prac.silver.metadata_table""").first()["last_processed_date"]

silver_df = spark.sql("""
                      select * from synth_prac.bronze.bronze_orders;""")

quarantined_df = silver_df.filter(F.col("amount")<0)

quarantined_df.write.format("delta").mode("append").saveAsTable("synth_prac.silver.quarantined_orders")

silver_df = silver_df.filter(F.col("order_date")>last_processed)

silver_df = silver_df.dropDuplicates(subset = ["order_id"])\
    .withColumn("order_date", F.to_date("order_date", "yyyy-MM-dd"))\
        .withColumn("amount", F.round(F.col("amount"),2))\
            .withColumn("payment_mode", F.upper(F.trim(F.col("payment_mode"))))\
                .filter(F.col("amount")>=0)

In [0]:
silver_df.createOrReplaceTempView("silver_orders_temp")

In [0]:
%sql
create table synth_prac.silver.silver_cleaned_orders_final(
  order_id bigint,
  customer_id bigint,
  product_id bigint,
  amount double,
  payment_mode string,
  order_date date,
  ingested_time timestamp
)
using delta;

In [0]:
%sql
merge into synth_prac.silver.silver_cleaned_orders_final f
using silver_orders_temp t
on t.order_id = f.order_id
when matched then update set *
when not matched then insert *

In [0]:
%sql
select * from synth_prac.silver.silver_cleaned_orders_final;

create or replace table synth_prac.silver.metadata_table as
select max(order_date) as last_processed_date
from synth_prac.silver.silver_cleaned_orders_final;

In [0]:
%sql
select * from synth_prac.silver.metadata_table;

In [0]:
top_customers_df = spark.sql("""select customer_id, round(total_amount,2) as total_amount,
                                round(1.0* total_amount/total_orders,2) as AOV, rn as rank
                                from(select customer_id, total_orders, total_amount,
                                        row_number() over(order by total_amount desc, total_orders) as rn
                                     from(select customer_id, count(*) as total_orders,
                                                round(sum(amount),2) as total_amount
                                         from synth_prac.silver.silver_cleaned_orders_final
                                         group by customer_id)t)r
                                where rn<=5""")

In [0]:
top_customers_df.createOrReplaceTempView("top_cust_view")

In [0]:
%sql
merge into synth_prac.gold.top_customers g
using top_cust_view t
on t.customer_id = g.customer_id
when matched then update set *
when not matched then insert*;

In [0]:
%sql
select * from synth_prac.gold.top_customers;

In [0]:
%sql
describe history synth_prac.gold.top_customers;

In [0]:
%sql
select * from synth_prac.gold.top_customers version as of 1;